# Aula 3 + Filtro Gaussiano — Máscaras e composição de imagens

**Aluno/Identificação:** CP l - APPLIED COMPUTER VISION

Nesta atividade serão combinados o filtro de média (`Blur`), o filtro gaussiano (`GaussianBlur`) e duas máscaras circulares de tamanhos diferentes.

In [ ]:
# Bibliotecas
import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titlesize'] = 12

## 1. Carregamento e preparação da imagem

A imagem é carregada pelo OpenCV em formato BGR e convertida para RGB apenas no momento da exibição.

In [ ]:
# Caminho da imagem no Google Colab
caminho_imagem = '/content/lenna.jpg'

imagem_bgr = cv2.imread(caminho_imagem)
if imagem_bgr is None:
    raise FileNotFoundError(
        f'Não foi possível carregar a imagem em: {caminho_imagem}. '
        'Verifique se o arquivo lenna.jpg foi enviado para o Colab.'
    )

imagem = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)
altura, largura = imagem.shape[:2]
centro = (largura // 2, altura // 2)

print(f'Dimensões da imagem: {largura} x {altura} pixels')

plt.imshow(imagem)
plt.title('Imagem original')
plt.axis('off')
plt.show()

## 2. Aplicação dos filtros

- `GaussianBlur` é aplicado à imagem inteira.
- `blur` é aplicado à imagem inteira com filtro de média.

In [ ]:
# Parâmetros dos filtros
kernel = (5, 5)

# Etapa 1: imagem inteira com filtro gaussiano
imagem_gaussiana = cv2.GaussianBlur(imagem, kernel, 0)

# Etapa 2: imagem inteira com filtro de média (Blur)
imagem_blur = cv2.blur(imagem, kernel)

fig, eixos = plt.subplots(1, 3)
for eixo, figura, titulo in zip(
    eixos,
    [imagem, imagem_gaussiana, imagem_blur],
    ['Original', 'GaussianBlur', 'Blur']
):
    eixo.imshow(figura)
    eixo.set_title(titulo)
    eixo.axis('off')
plt.tight_layout()
plt.show()

## 3. Máscaras circulares

A máscara menor será aplicada à imagem original. A máscara maior será aplicada à imagem com `GaussianBlur`. Os raios são calculados a partir do menor lado da imagem para manter o código adaptável a diferentes resoluções.

In [ ]:
# Raios distintos: o círculo gaussiano é maior que o círculo original
raio_menor = int(min(altura, largura) * 0.18)
raio_maior = int(min(altura, largura) * 0.32)

# Máscara do círculo menor
mascara_menor = np.zeros((altura, largura), dtype=np.uint8)
cv2.circle(mascara_menor, centro, raio_menor, 255, thickness=-1)

# Máscara do círculo maior
mascara_maior = np.zeros((altura, largura), dtype=np.uint8)
cv2.circle(mascara_maior, centro, raio_maior, 255, thickness=-1)

fig, eixos = plt.subplots(1, 2)
eixos[0].imshow(mascara_menor, cmap='gray')
eixos[0].set_title(f'Máscara menor (raio={raio_menor})')
eixos[1].imshow(mascara_maior, cmap='gray')
eixos[1].set_title(f'Máscara maior (raio={raio_maior})')
for eixo in eixos:
    eixo.axis('off')
plt.tight_layout()
plt.show()

## 4. Recortes circulares com máscara

Nesta etapa são preservadas somente as regiões dentro dos círculos. O restante da imagem fica preto.

In [ ]:
# Etapa 3: círculo menor recortado na imagem original
recorte_original = cv2.bitwise_and(imagem, imagem, mask=mascara_menor)

# Etapa 4: círculo maior recortado na imagem GaussianBlur
recorte_gaussiano = cv2.bitwise_and(imagem_gaussiana, imagem_gaussiana, mask=mascara_maior)

fig, eixos = plt.subplots(1, 2)
eixos[0].imshow(recorte_original)
eixos[0].set_title('Etapa 3 — círculo menor na imagem original')
eixos[1].imshow(recorte_gaussiano)
eixos[1].set_title('Etapa 4 — círculo maior no GaussianBlur')
for eixo in eixos:
    eixo.axis('off')
plt.tight_layout()
plt.show()

## 5. Composição sobre a imagem com Blur

A região do círculo maior, proveniente do `GaussianBlur`, é colocada sobre a imagem com `Blur`. Fora do círculo maior, a imagem com `Blur` permanece visível.

In [ ]:
# Etapa 5: GaussianBlur circular sobre a imagem Blur
composicao_blur_gaussiano = np.where(
    mascara_maior[..., np.newaxis] == 255,
    imagem_gaussiana,
    imagem_blur
).astype(np.uint8)

plt.imshow(composicao_blur_gaussiano)
plt.title('Etapa 5 — GaussianBlur sobre a imagem Blur')
plt.axis('off')
plt.show()

## 6. Composição final

O círculo menor da imagem original é colocado sobre o resultado da etapa 5. Assim, a composição final apresenta três regiões: a imagem original no centro menor, o `GaussianBlur` no anel circular maior e o `Blur` no fundo.

In [ ]:
# Etapa 6: círculo menor original sobre a composição da etapa 5
resultado_final = np.where(
    mascara_menor[..., np.newaxis] == 255,
    imagem,
    composicao_blur_gaussiano
).astype(np.uint8)

plt.figure(figsize=(8, 8))
plt.imshow(resultado_final)
plt.title(f'Resultado final — {nome}')
plt.axis('off')
plt.show()

## Resumo das etapas

1. Aplicação de `GaussianBlur` na imagem inteira.
2. Aplicação de `Blur` na imagem inteira.
3. Recorte circular menor da imagem original.
4. Recorte circular maior da imagem com `GaussianBlur`.
5. Sobreposição da etapa 4 sobre a imagem com `Blur`.
6. Sobreposição da etapa 3 sobre o resultado da etapa 5.